In [ ]:
# Import relevant modules
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.problems import get_problem
from pymoo.core.problem import Problem
from pymoo.optimize import minimize
from pymoo.visualization.scatter import Scatter
from pymoo.operators.crossover.pntx import TwoPointCrossover
from pymoo.operators.mutation.bitflip import BitflipMutation
from pymoo.operators.mutation.pm import PolynomialMutation
from pymoo.core.mixed import MixedVariableGA
from pymoo.core.mixed import StructuredCrossover # custom function
from pymoo.core.mixed import StructuredMutation # custom function
from pymoo.operators.sampling.rnd import FloatRandomSampling, BinaryRandomSampling
from pymoo.operators.selection.tournament import TournamentSelection 
from pymoo.operators.crossover.pcx import PCX
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.soo.nonconvex.ga import GA
from pymoo.core.variable import Real, Integer, Choice, Binary
from pymoo.algorithms.moo.nsga2 import RankAndCrowdingSurvival
from pymoo.operators.crossover.pntx import PointCrossover
from pymoo.core.repair import Repair
from datetime import datetime, timezone
from pymoo.util.display.multi import MultiObjectiveOutput
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.core.mixed import MixedVariableSampling
from pymoo.core.mixed import MixedVariableMating
from pymoo.core.mixed import MixedVariableDuplicateElimination
from pymoo.termination.default import DefaultMultiObjectiveTermination
from datetime import timedelta

import polars as pl
import calendar
import random
import math
import numpy as np
import warnings
import sys
import os
import pandas as pd
import shutil

from pymoo.config import Config
Config.warnings['not_compiled'] = False

# Ensure PowerFactory root is in PATH, ensure PowerFactory and System python are same version, PowerFactory must be closed
# Point to the .pyd folder
sys.path.append(r"C:\Program Files\DIgSILENT\PowerFactory 2026 SP1\Python\3.14")

import powerfactory as pf 

In [ ]:
# Activate PowerFactory project and case study

project_name = "My project name" # define PowerFactory project name
study_case_name = "My case study name"   # Define PowerFactory case study name


app = pf.GetApplication()

if app is None:
    raise Exception("Could not connect to PowerFactory!")
else:
    print("Connected successfully to PowerFactory:", app)
      
user = app.GetCurrentUser()
projects = user.GetContents("*.IntPrj")  # get all projects under user

# Find project by name and activate
target_project = None

for proj in projects:
    if proj.loc_name == project_name:
        target_project = proj
        break

if target_project is None:
    raise Exception(f"Project '{project_name}' not found!")

target_project.Activate()
print(f"Activated Project: {target_project.loc_name}")

study_case_folder = app.GetProjectFolder("study")  # gets the Study Cases folder
study_cases = study_case_folder.GetContents("*.IntCase")

# Find study case by name and activate

target_sc = None

for sc in study_cases:
    if sc.loc_name == study_case_name:
        target_sc = sc
        break

if target_sc is None:
    raise Exception(f"Study case '{study_case_name}' not found!")

target_sc.Activate()
print(f"Activated Study Case: {target_sc.loc_name}")

In [ ]:
# Set parameters and paths for the project

START_TIME      = datetime(2030, 1, 1,  0, 0, 0)   # simulation start date format [YYYY,MM,DD,  HH,MM,SS]
END_TIME        = datetime(2051, 1, 1,  0, 0, 0)   # simulation end date format [YYYY,MM,DD,  HH,MM,SS]
STEP_SIZE       = 1    # step size value for the STEP_UNIT value
STEP_UNIT       = 2    # 0=seconds 1=minute  2=hour  3=day 4=months 5=years
CALC_PERIOD     = 4    # 0=complete day 1=complete month 2=complete year 4=User defined time range determined by variable START_TIME and END_TIME
IOPT_NET        = 0    # 0=AC balanced , 1=AC unbalanced, 2=DC 
INIT_TYPE       = 1    # 0=based on previous load flow results | 1=Full initialisation
I_CONS_TIME_DEP = 0    # 1=ON: consider time-dependent states of models | 0=OFF

grid = 'KENYA' # The name of the grid under study in PowerFactory

opex_rate = 0.3 # Opex as a fraction of Capex applied in the project

bess_capex = 192000 # CAPEX for BESS systems in $/MWh
c_gen = 30 # Cost of generation in $/MW. If using different values for generation, create a dictionary mapping instead.
cost_ens = 5000 # Value of Lost Load or Energy Not Supplied

cost_curt_unit = 64 # The cost of curtailing generation in $/MW

v_lower = 0.95 # Lower voltage limit for terminals- ensure this is the same applied in your powerfactory project
v_upper = 1.05 # Upper voltage limit for terminals

ELEMENT_MAP = {
    # It maps the various parameters to be compiled later in calculation of intermediate results. This is optional
    "df_grid" : {
        "class"    : "*.ElmNet",
        "variables": ["c:TotgenP", "c:TotgenQ", "c:LoadPnom", "c:LoadQnom","c:LoadPdem", "c:NetQ", "c:LossP", "c:LossQ", "c:GenPdisp",
                      "c:TotconsP", "c:TotconsQ", "c:SupP", "c:UnsupP", "c:NrTermMin", "c:NrTermMax", "c:NrOvlBranch", "c:GenQdiff",
                     "c:Maxloading", "c:umin", "c:umax", "c:GenPmism"],
    },
  
}

''' 
Define ChaTime objects from powerfactory
ChaTime objects in PowerFactory can be used to define time characteristics. 
In this case we use it to define 24 hour load profile curves.
The ChaTime objects have to be created in PowerFactory before running the script.
'''
EV_time = "Time Characteristic EV" # the name of the Electric vehicle ChaTime defined as its 'loc_name' in PowerFactory
IND_time = "Time Characteristic Ind" # the name of the industrial load ChaTime defined as its 'loc_name' in PowerFactory
RES_time = "Time Characteristic Dom" # the name of the residential load ChaTime defined as its 'loc_name' in PowerFactory


''' Define base values used to determine base load profile [a, b, c]
        base_a >> hours 00–06 and hour 23
        base_b >> # hours 07–17
        base_c >>  # hours 18–22 
        The values are defined as ratio of peak in per unit
        '''
res_base_a, res_base_b, res_base_c = 0.5, 0.7, 1.0
ev_base_a, ev_base_b, ev_base_c = 0.3, 0.4, 1.0
ind_base_a, ind_base_b, ind_base_c = 0.2, 1.0, 0.2


''' csv paths
This section assumes the storage and generation units are already created in PowerFactory. 
Supply the names in this section as a CSV or list

'''
storage_list = pl.read_csv(r"C:\Users\dorcus70\Documents\GA code\storage.csv") # or storage_list = [name 1, name 2]
storage = storage_list[:, 0].to_list() # skip if using list instead of csv

generator_file = pl.read_csv(r"C:\Users\dorcus70\Documents\GA code\generators to add.csv") 
''' format csv with column names: gen_name and total. 
gen_name is the name equivalent to its 'loc_name' in PowerFactory, and total is the total cost of installing this generator.'''

powerplants = generator_file["gen_name"].to_list() 
powerplants_cost = dict(
    zip(generator_file["gen_name"], generator_file["total"])
)

decomm = pl.read_csv(r"C:\Users\dorcus70\Documents\GA code\decom_gen.csv") 
''' optional: if any generator is to be decommissioned during the
period of analysis. column names to use: loc_name and year. year is when the item is to be decommissioned.'''

print("Done")

In [ ]:
# Define intermediate functions to be used in the optimisation

def build_vector(value_a, value_b, value_c):
    '''
    Accepts the 3 input values and generates a 24 hour vector (bins)

    Accepts [a,b,c] and returns [a,a,a,a,a,a,a,b,b,b,b,b,b,b,b,b,b,b,c,c,c,c,c,a]
    '''
    return (
        [value_a] * 7  +   # hours 00–06
        [value_b] * 11 +   # hours 07–17
        [value_c] * 5  +   # hours 18–22
        [value_a] * 1      # hour 23
    )

def compute_target_sum(value_a, value_b, value_c):
    """Fit a cubic poly to the BASE values and sum the smoothed curve.
    Used for the comparison to new generated values from the algorithm

    """
    t = np.arange(24)
    p_base = np.array(build_vector(value_a, value_b, value_c))
    coeffs = np.polyfit(t, p_base, 3)
    p_smooth_base = np.polyval(coeffs, t)

    return p_smooth_base.sum()

# Compute base load targets once
TARGET_SUM_RES = compute_target_sum(res_base_a, res_base_b, res_base_c) # Residential target computed from base
TARGET_SUM_EV = compute_target_sum(ev_base_a, ev_base_b, ev_base_c) # EV target computed from base
TARGET_SUM_IND = compute_target_sum(ind_base_a, ind_base_b, ind_base_c) # Industrial target computed from base

def fit_constrained_poly(new_a, new_b, new_c, target_sum):
    """
    Fit a cubic polynomial to the new values,
    constrained so that sum(p_smooth) == target_sum.

    It converts the base curves received in step curve format to a third-degree polynomial    
    """
    t = np.arange(24)
    
    p_new = np.array(build_vector(new_a, new_b, new_c))
    
    def objective(coeffs):
        return np.sum((np.polyval(coeffs, t) - p_new) ** 2)

    def sum_constraint(coeffs):
        return np.polyval(coeffs, t).sum() - target_sum  # == 0

    from scipy.optimize import minimize as mnze
    
    coeffs_init = np.polyfit(t, p_new, 3)
    result = mnze(
        objective,
        coeffs_init,
        constraints={'type': 'eq', 'fun': sum_constraint},
        method='SLSQP'
    )
    return np.polyval(result.x, t).tolist()


def update_timechar(updates):
    '''
    This function will update ChaTime objects with the new values from the GA algorithm.
    ── Define your ChaTime updates, update ChaTime objects in PowerFactory to control load curves
    Format: 'ChaTime name': (VALUE_A, VALUE_B, VALUE_C)
      Band A — OFF-PEAK night  : 00:00 – 06:59  → VALUE_A
      Band B — STANDARD day    : 07:00 – 17:59  → VALUE_B
      Band C — PEAK evening    : 18:00 – 22:59  → VALUE_C
      Band D — OFF-PEAK night  : 23:00 – 24:00  → VALUE_A  (same as A)
    '''
  
    #  Fetch all ChaTime objects once 
    chatim_objects = app.GetCalcRelevantObjects('*.ChaTime')
    if chatim_objects is None:
        raise Exception("No ChaTime object not found in project")    
    chatim_by_name = {o.loc_name: o for o in chatim_objects}

    df_updates = pl.DataFrame([
        {
            "name": name,
            "band_values": ";".join(map(str, band_values))
        }
        for name, band_values in updates.items()
    ])
   
    #  Loop and update
    for name, band_values in updates.items():
        
        # Validate band values provided
        if len(band_values) != 24:
            print(f"   ERROR: expected 24, got {band_values} — skipping.\n")
            continue
    
        # Build the 24-hour vector
        new_vector = band_values

        # Find the ChaTime object
        cha = chatim_by_name.get(name)
        if cha is None:
            print(f"   ERROR: '{name}' not found in project — skipping.\n")
            continue
            
        cha.SetAttribute('vector', new_vector)
        cha.WriteChangesToDb()

def update_equip(current_year): 
    
    # this updates the out-of-service values for generators and storage in each year of analysis
    # Checks if current year is before the installation year and sets it in or out of service

    df_add_to_pf = pl.DataFrame(add_to_pf)

    grid = next((g for g in app.GetCalcRelevantObjects("ElmNet") if g.loc_name == grid), None) 
    errors = []
    if grid is None:
        raise Exception("Grid not found")
    
    for row in df_add_to_pf.to_dicts():
        loc_name  = str(row["loc_name"])
        size      = row["size"]
        pf_class  = str(row["type"])
        year  = str(row["year"])
        bess_yn = str(row["bess_yn"])
        dsl = str(row["dsl"])
        charge = row["charge"]
        discharge = row["discharge"]
        
        try:
            # find element
            if bess_yn == "1": # storage                
                model = next((m for m in app.GetCalcRelevantObjects("*.ElmGenstat") if m.loc_name == loc_name), None)
                if model is None:
                    raise Exception(f"Failed to find '{loc_name}'")
                if int(current_year) >= int(year):
                    model.outserv  = 0
                else:
                    model.outserv  = 1            
            else: # gen
                model = next((m for m in app.GetCalcRelevantObjects("*.ElmSym") if m.loc_name == loc_name), None)
                if model is None:
                    raise Exception(f"Failed to find '{loc_name}'")
                if int(current_year) >= int(year):
                    model.outserv  = 0
                else:
                    model.outserv  = 1                       
                    
        except Exception as e:
            print(f"❌ Error on update_equip for {loc_name}: {e}")
          
    # update decommissioned generators (optional) 
    for row in decomm.to_dicts():
        loc_name  = str(row["loc_name"])
        year  = str(row["year"])
        
        try:             
            # find element
            model = next((m for m in app.GetCalcRelevantObjects("*.ElmSym") if m.loc_name == loc_name), None)
            if model is None:
                raise Exception(f"Failed to find '{loc_name}'")
            if int(current_year) >= int(year):
                model.outserv  = 1
            else:
                model.outserv  = 0            

        except Exception as e:
            print(f"❌ Error on update_equip for {loc_name}: {e}")
   
    
def updatepf():
    # update storage models with the new parameters for each solution. These are ElmQdsl elements in PowerFactory
    # Also update the associated static generators apparent power
    
    df_add_to_pf = pl.DataFrame(add_to_pf)

    grid = next((g for g in app.GetCalcRelevantObjects("ElmNet") if g.loc_name == f"{grid}"), None)
    
    if grid is None:
        raise Exception("Grid not found")
    
    for row in df_add_to_pf.to_dicts():
        loc_name  = str(row["loc_name"])
        size      = row["size"]
        pf_class  = str(row["type"])
        year  = str(row["year"])
        bess_yn = str(row["bess_yn"])
        dsl = str(row["dsl"])
        charge = row["charge"]
        discharge = row["discharge"]
        
        try:
            # 2. find element
            if bess_yn == "1": # storage
                # 2. find element
                if (size > 0):
                    model = next((m for m in app.GetCalcRelevantObjects("*.ElmQdsl") if m.loc_name == dsl), None)
                    if model is None:
                        raise Exception(f"Failed to find '{loc_name}'")
                        
                    model.outserv  = 0 # ensure it is in service
                    model.Eini  = size # capacity in MWh
                    model.Pstore  = charge * size # Charge rate (active power)
                    model.Qstore  = charge * size * 0.3 # change this if using different value for reactive power as fraction of active power
                    model.Pfeed  = discharge * size # feed in power
                    model.Qfeed  = discharge * size * 0.3

                    model2 = next((m for m in app.GetCalcRelevantObjects("*.ElmGenstat") if m.loc_name == loc_name), None)
                    if model2 is None:
                        raise Exception(f"Failed to find '{loc_name}'")                        
                    model2.sgn  = ((charge * size) + (charge * size * 0.3)) # apparent power
                    
        except Exception as e:
            print(f"❌ Error on update_pf for {loc_name}: {e}")
   
      
def update_out_of_service():
    # This enforces the out-of-service for storage and generators that the algorithm excludes from the optimisation
    
    delete = pl.DataFrame(for_deletion)

    grid = next((g for g in app.GetCalcRelevantObjects("ElmNet") if g.loc_name == grid), None)
    errors = []
    if grid is None:
        raise Exception("Grid not found")
    
    for row in delete.to_dicts():
        loc_name  = str(row["name"])
        bess_yn = str(row["bess_yn"])
        
        try:
            # 2. find element
            if bess_yn == "1": # storage                
                # find element
                model = next((m for m in app.GetCalcRelevantObjects("*.ElmGenstat") if m.loc_name == loc_name), None)
                if model is None:
                    raise Exception(f"Failed to find '{loc_name}'")
                model.outserv  = 1           
            else: # gen
                # find element
                model = next((m for m in app.GetCalcRelevantObjects("*.ElmSym") if m.loc_name == loc_name), None)
                if model is None:
                    raise Exception(f"Failed to find '{loc_name}'")
                model.outserv  = 1                       

        except Exception as e:
            print(f"❌ Error on {loc_name}: {e}")  
            
def loadflow1(X):

    '''Run the load flow and compile for each year of analysis'''
    
    global converge, ineq_dispatch, ineq_qdispatch, total_steps, total_deviations, other_cost

    # Initiate variables
    year_num = 0
    total_deviations = 0
    other_cost = 0
    state = {
        "non_conv_total": 0,
        "total_losses": 0,
        "total_gen": 0,
        "total_gen_costs": 0,
        "total_load": 0,
        "total_ENSs": 0,
        "total_ens_costs": 0,
        "total_available": 0,
        "total_from_storage": 0,
        "total_curtailed": 0,
        "total_curtailed_cost": 0,
        "total_lower_dev": 0,
        "total_upper_dev": 0,
        "total_qmiss": 0
    }

    umax = -999
    umin = 999
    maxl = -999 

    converge = 99
    ineq_dispatch = 99
    ineq_qdispatch = 99  

    total_steps = 0
    
    # reset calc before each load flow
    app.ResetCalculation()
    
    for year in range(START_TIME.year, END_TIME.year):

        # Iterate through multiple years to ensure we can update equipment in each year

        month = random.randint(1, 12)
        days_in_month = calendar.monthrange(year, month)[1]
        
        day = random.randint(1, days_in_month)
        
        start = datetime(year, month, day, 0, 0, 0)
        end = start + timedelta(days=1)

        year_num += 1
    
        # 1. adjust all out of service 
        update_equip(year)        

        # Run simulation with user defined parameters
        qds = app.GetFromStudyCase("ComStatsim")
        if qds is None:
            raise Exception("ComStatsim not found in study case")
        
        # user-defined parameters
        qds.startTime        = int(start.timestamp())
        qds.endTime          = int(end.timestamp())
        qds.stepSize         = STEP_SIZE
        qds.stepUnit         = STEP_UNIT
        qds.calcPeriod       = CALC_PERIOD
        qds.iopt_net         = IOPT_NET
        qds.iConsTimeDepMod  = I_CONS_TIME_DEP

        if year_num == 1: # full initialisation
            qds.initType         = 1
        else:
            qds.initType         = 0 # use previous results (optional)

        result = qds.Execute() # initiate quasi dynamic simulation

        if result == 0:
            res = qds.results
            if res is None:
                raise Exception("No results object found!")
        
            res.Load()

            n_timesteps = app.ResGetValueCount(res, 0)

            total_steps += n_timesteps
        
            #  Simulation-level columns (on the result object itself)
            sim_obj = res.GetObject(0)
        
            def get_sim_col(obj, var):
                """Return a callable that fetches var for a given step via ElmRes.FindColumn."""
                col = res.FindColumn(obj, var)   # replaces ResGetIndex
                return col  # store the column index; 
        
            col_tstep  = res.FindColumn(sim_obj, "b:number")
            col_noconv = res.FindColumn(sim_obj, "b:noconv")
            col_time   = res.FindColumn(sim_obj, "b:ucttime")
        
            def fetch(step, col):
                """Thin wrapper: returns the float value or None on error."""
                err, val = res.GetValue(step, col)
                return val if err == 0 else None
        
            # ---------- Build DataFrames ----------
            OUT_OF_SERVICE_CLASSES = {"*.ElmSym", "*.ElmGenstat", "*.ElmTerm"}
            dataframes = {}
            
            for df_name, config in ELEMENT_MAP.items():
                cls, variables = config["class"], config["variables"]
                elements = app.GetCalcRelevantObjects(cls)
            
                if not elements:
                    dataframes[df_name] = pl.DataFrame()
                    continue
            
                filter_oos = cls in OUT_OF_SERVICE_CLASSES
            
                el_col_cache = {
                    el: {var: res.FindColumn(el, var) for var in variables}
                    for el in elements
                    if not filter_oos or el.outserv == 0
                }
            
                all_data = []
                for el, var_cols in el_col_cache.items():
                    for step in range(n_timesteps):
                        row = {
                            "Name"     : el.loc_name,
                            "Step"     : step,
                            "b:number" : int(fetch(step, col_tstep)  or 0),
                            "b:noconv" : int(fetch(step, col_noconv) or 0),
                            "b:time"   : datetime.fromtimestamp(
                                             fetch(step, col_time), tz=timezone.utc
                                         ),
                        }
                        for var, col in var_cols.items():
                            row[var] = fetch(step, col) if col >= 0 else None
            
                        all_data.append(row)
            
                df = pl.DataFrame(all_data)  
                dataframes[df_name] = df
               
            df_grid = dataframes["df_grid"]

            # ---------- Zone Aggregation ----------
            ZONE_CONFIG = {
                "grid": {
                    "df"      : df_grid,
                    "name_val": grid,
                    "keys"    : ("umax",   "umin",   "maxl"),
                }, # can define more zones if needed
            }
        
            # Accumulators: keyed by the variable names in ZONE_CONFIG
            accumulators = {
                "grid": [umax,      umin,      maxl     ],
            } # can add more zones if needed
        
            for zone_key, cfg in ZONE_CONFIG.items():
                df      = cfg["df"]
                name    = cfg["name_val"]
                uk, lk, ml = cfg["keys"]

                   # ensure df exists and has required columns
                required_cols = ["Name", "b:noconv"]
                
                if (
                    df is None
                    or df.is_empty()
                    or any(col not in df.columns for col in required_cols)
                ):
                    continue           
                    
                mask = (
                    (pl.col('Name') == name) &
                    (pl.col('b:noconv') == 0) &
                    (pl.col('c:umin') != 0)
                )
                
                mask_df = (
                    (pl.col('Name') == name) &
                    (pl.col('b:noconv') == 0)
                )
                sub = df.filter(mask)
                sub_loading = df.filter(mask_df)
                
                if sub.is_empty():
                    continue
        
                cur_max, cur_min, cur_ml = (
                    sub["c:umax"].max(),
                    sub["c:umin"].min(),
                    sub_loading["c:Maxloading"].max(),
                )
        
                acc = accumulators[zone_key]
                if cur_max > acc[0]: acc[0] = cur_max
                if cur_min < acc[1]: acc[1] = cur_min
                if cur_ml  > acc[2]: acc[2] = cur_ml
        
            # Unpack grid accumulators back to named variables
            umax, umin, maxl = accumulators["grid"]

            # ---------- Cost / Deviation ----------
            other_cost       += loopcost(year, df_grid, X, state)

            total_deviations += loopdev(year, df_grid, X, state)

        else:
            other_cost       = 1e50 # very high penalty for result == 1, infeasible solution
            total_deviations = 1e6 # very high penalty
        
        # ---------- State → X ---------- 
        # save intermediate results to algorithms X 
        converge      = state["non_conv_total"]
        ineq_qdispatch= state["total_qmiss"]
        
        STATE_KEYS = [
            "_no_conv_count",  "_total_losses",     
            "_total_gen",      "_total_gen_costs",   "_total_load",
            "_total_ENSs",     "_total_ens_costs",   "_total_available",
            "_total_from_storage", "_total_curtailed", "_total_curtailed_cost",
            "_total_lower_dev","_total_upper_dev",
        ]
        STATE_MAP = {
            "_no_conv_count"       : "non_conv_total",
            "_total_losses"        : "total_losses",
            "_total_gen"           : "total_gen",
            "_total_gen_costs"     : "total_gen_costs",
            "_total_load"          : "total_load",
            "_total_ENSs"          : "total_ENSs",
            "_total_ens_costs"     : "total_ens_costs",
            "_total_curtailed"     : "total_curtailed",
            "_total_curtailed_cost": "total_curtailed_cost",
            "_total_lower_dev"     : "total_lower_dev",
            "_total_upper_dev"     : "total_upper_dev",
        }
        for x_key, state_key in STATE_MAP.items():
            X[x_key] = state[state_key]
        
        # Zone voltages/loading → X
        ZONE_X_MAP = {
            ("_umax",      "_umin",      "_Maxloading") : "grid",
        }
        for (xmax, xmin, xml), zone_key in ZONE_X_MAP.items():
            acc = accumulators[zone_key]
            X[xmax], X[xmin], X[xml] = acc[0], acc[1], acc[2]


def constraints():
    # returns constraint violation results
    return converge, total_steps

    
def loopcost(year, df_grid, X, state):
    # Calculate costs

    # loss
    mask = (pl.col('Name') == grid) & (pl.col('b:noconv') == 0) # consider converged cases only, non-converged cases are penalised
    df_kenya = df_grid.filter(mask)
    
    losses = df_kenya['c:LossP'].sum()

    # count non converged timesteps
    non_conv = df_grid.filter(
        (pl.col('Name') == grid) &
        (pl.col('b:noconv') == 1)
    ).height
    
    # Generation
    gen = df_kenya.select(
        pl.min_horizontal('c:TotgenP', 'c:GenPdisp').sum() # choose which metric works best for generation 
    ).item()    
    
    total_gen_cost = c_gen * gen
    
    # ENS
    load = df_kenya['c:LoadPdem'].sum()
    
    ens = (losses + load) - gen
    if ens < 0:
        ens = 0
    total_ens = float(cost_ens) * ens
    
    # Curtailment
    curt_gen = df_kenya.filter(
        pl.col('c:GenPmism') < 0
    )['c:GenPmism'].abs().sum()
    
    if curt_gen < 0:
        curt_gen = 0
    
    # Unmet reactive power
    genq = df_kenya['c:GenQdiff'].sum()
    
    total_cost_curt = curt_gen * cost_curt_unit

    # save values
    state["non_conv_total"] += non_conv
    state["total_losses"] += losses
    state["total_gen"] += gen
    state["total_gen_costs"] += total_gen_cost
    state["total_load"] += load
    state["total_ENSs"] += ens
    state["total_ens_costs"] += total_ens
    state["total_curtailed"] += curt_gen
    state["total_curtailed_cost"] += total_cost_curt
    state["total_qmiss"] += genq
    
    return total_ens + total_gen_cost + total_cost_curt

  
def loopdev(year, df_grid, X, state):
    # calculate total deviations
    
    # Filter for lower and upper deviations and b:noconv == 0
    mask = (pl.col('Name') == grid) & (pl.col('b:noconv') == 0)
    df_kenya = df_grid.filter(mask)
    
    # Sum deviations
    total_lower = df_kenya['c:NrTermMin'].sum() # minimum deviations
    total_upper = df_kenya['c:NrTermMax'].sum() # maximum deviations
    total_deviations = total_lower + total_upper

    # save values
    state["total_lower_dev"] += total_lower
    state["total_upper_dev"] += total_upper
    
    return total_deviations


def cost(self, X):

    # extract additions for powerfactory    
    global add_to_pf, for_deletion

    # initialise variables
    for_deletion = []
    add_to_pf = []
    updates = {}
    
    total_investment_plant = 0 # bess + generation
    total_investment_bess = 0
    total_opex_plant = 0
    total_opex_bess = 0
    updates_gen = {}
    
    # check for components being added BESS/Powerplants
    for k in self.var_keys:
        if k.startswith(("xab", "xas")) and X[k]:
            suffix = k[3:]  # remove xab or xas    
            if k.startswith("xab"):
                year_key = "yab" + suffix
                add_to_pf.append({
                    "loc_name": k[4:],
                    "dsl": "dsl" + suffix,
                    "year": X.get(year_key),
                    "bess_yn": 0,
                    "type": "ElmQdsl",
                })    
                updates_gen["loc_name"] =  X.get(year_key)

                plant = k[4:]
                plant_inv = powerplants_cost[f"{plant}"]
                total_investment_plant += plant_inv # calculate cost
                year_datetime = datetime(X.get(year_key), 1, 1, 0, 0, 0)
                years = ((END_TIME - year_datetime).days) / 365.0
                total_opex_plant += (plant_inv * opex_rate * years)
          
            else:  # startswith xas
                size_key = "sas" + suffix    
                charge_key = "cas" + suffix    
                dis_key = "das" + suffix   
                year_key = "yas" + suffix
                bess_inv = bess_capex * X.get(size_key)
                year_datetime = datetime(X.get(year_key), 1, 1, 0, 0, 0)
                years = ((END_TIME - year_datetime).days) / 365.0
                total_investment_bess += bess_inv # calculate cost
                total_opex_bess += (bess_inv * opex_rate * years)
                updates_gen["dsl2" + suffix] =  X.get(year_key)
                add_to_pf.append({
                    "loc_name": k[4:], # >>>>>>>>>>>>>>BESS OR ITS MODEL NAME
                    "dsl": "dsl" + suffix, # fisrt dynamic model for storage
                    "size": X.get(size_key),
                    "bess_yn": 1,                    
                    "type": "ElmQdsl",
                    "charge": X.get(charge_key),
                    "discharge": X.get(dis_key),
                    "year": X.get(year_key),
                    
                })    

    # Double check data to ensure False is reinforced
    for k in self.var_keys:
        if k.startswith(("xab", "xas")) and not X[k]:
            if k.startswith("xab"):
                for_deletion.append({
                    "name": k[4:],
                    "bess_yn": "0",
                })  
            else:  # startswith xas
                for_deletion.append({
                    "name": k[4:], 
                    "bess_yn": "1",
                }) 
                
    total_opex = total_opex_plant + total_opex_bess
    total_investment = total_investment_plant + total_investment_bess   

    # save values
    X["_total_opex_plant"] = total_opex_plant
    X["_total_opex_bess"] = total_opex_bess
    X["_total_opex"] = total_opex
    X["_total_investment_plant"] = total_investment_plant
    X["_total_investment_bess"] = total_investment_bess    
    X["_total_investment"] = total_investment
    
    # fit the load profiles into a polynomial 
    for k in self.var_keys:
        if k.startswith("EV"):
            night_key = "EV_night"
            offpeak_key = "EV_offpeak"
            peak_key = "EV_peak"
            updates[f"{EV_time}"] = (fit_constrained_poly(X.get(night_key), X.get(offpeak_key), X.get(peak_key), TARGET_SUM_EV))
                                                         
            
        elif k.startswith("RES"):
            night_key = "RES_night"
            offpeak_key = "RES_offpeak"
            peak_key = "RES_peak"
            updates[f"{RES_time}"] = (fit_constrained_poly(X.get(night_key), X.get(offpeak_key), X.get(peak_key), TARGET_SUM_RES))
            
        elif k.startswith("IND"):
            night_key = "IND_night"
            offpeak_key = "IND_offpeak"
            peak_key = "IND_peak"
            updates[f"{IND_time}"] = (fit_constrained_poly(X.get(night_key), X.get(offpeak_key), X.get(peak_key), TARGET_SUM_IND))
            
    X["_EV_curve"] = updates[f"{EV_time}"] 
    X["_RES_curve"] = updates[f"{RES_time}"]    
    X["_IND_curve"] = updates[f"{IND_time}"]
                 
    # update ChaTime >>> time characteristics of load
    update_timechar(updates)

    # update powerfactorymodel to include bess and capacity expansion
    updatepf()

    # enforce out of service for relevant equipment
    update_out_of_service()

    # initiate quasi-dynamic simulation
    loadflow1(X)   
    
    return round(total_investment + total_opex + other_cost, 2)
    
    
def dev(self, X):
    # return result of objective 2
    return round(total_deviations, 2)
    
print("done")

In [ ]:
class MultiObjectiveMixedVariableProblem(ElementwiseProblem):

    def __init__(self, **kwargs):

        vars = {
        }
         # EV load decision variables
        vars["EV_night"]  = Real(bounds=(0.3, 0.5))             
        vars["EV_offpeak"]  = Real(bounds=(0.3, 0.5)) 
        vars["EV_peak"]  = Real(bounds=(0.5, 1)) 

        # Ind load decision variables
        vars["IND_night"]  = Real(bounds=(0.5, 0.7))             
        vars["IND_offpeak"]  = Real(bounds=(0.6, 1)) 
        vars["IND_peak"]  = Real(bounds=(0.5, 0.7))         

         # Res load decsision variables
        vars["RES_night"]  = Real(bounds=(0.5, 0.7))             
        vars["RES_offpeak"]  = Real(bounds=(0.7, 0.9)) 
        vars["RES_peak"]  = Real(bounds=(0.5, 1))   

        # Capacity expansion decision variables
        # suffixes are defined to easily process the related variables in the intermediate calculations
        for powerplant in powerplants:
            vars[f"xab_{powerplant}"] = Binary() #choice to add powerplant
            vars[f"yab_{powerplant}"]  = Integer(bounds=(2030, 2050)) #year to add powerplant

            
        # BESS to existing VER decision variables
        for s in storage:
            vars[f"xas_{s}"] = Binary() #choice to add bess
            vars[f"sas_{s}"]  = Choice(options=list(range(50, 301, 50))) #size of bess
            vars[f"cas_{s}"]  = Choice(options=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1]) #charge rate
            vars[f"das_{s}"]  = Choice(options=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1]) #discharge rate
            vars[f"yas_{s}"]  = Integer(bounds=(2030, 2050)) #year to add powerplant

        
        # Store keys for auto-unpacking later
        self.var_keys = list(vars.keys())   
                
        super().__init__(vars=vars, n_obj=2, n_ieq_constr=1, n_eq_constr=0, **kwargs)
        

    def _evaluate(self, X, out, *args, **kwargs):
        
        # unpack variables
        for k in self.var_keys:
            setattr(self, k, X[k])
                
        f1 = cost(self, X) # first objective- cost
        f2 = dev(self, X) # second objective - voltage deviations

        constraints()

        if converge > 0: # add penalty if applicable
            f1 = f1 + (f1 * converge)
            f2 = f2 + (f2 * converge)
            
        out["F"] = [f1, f2]
                    
        
problem = MultiObjectiveMixedVariableProblem()

termination = DefaultMultiObjectiveTermination(
    xtol=1e-8,       # design space tolerance
    cvtol=1e-6,      # constraint violation tolerance  
    ftol=0.0025,     # objective space tolerance (key for "no improvement")
    period=15,       # number of generations to look back- algorithm must run this many generations before check starts
    n_max_gen=65,   # max generations as safety net
    n_max_evals=10000 # max evaluations as safety net
)


algorithm = NSGA2(
    pop_size=50,
    sampling=MixedVariableSampling(),
    crossover=StructuredCrossover(prob=0.9),
    mutation=StructuredMutation(),
    survival=RankAndCrowdingSurvival(crowding_func='pcd'),
    output=MultiObjectiveOutput(),

    mating=MixedVariableMating(
        eliminate_duplicates=MixedVariableDuplicateElimination()
    ),

    eliminate_duplicates=MixedVariableDuplicateElimination()  # 
)


print("\nStarting optimisation:", datetime.now())
algorithm.termination = termination
res = minimize(problem,
               algorithm,
               seed=1,
               save_history=True,
               termination=termination,
               verbose=True)


print("\nCompleted optimisation:", datetime.now())

In [ ]:
# Save results per generation and population (optional)
rows = []

def to_scalar(x):
    if x is None:
        return None
    try:
        return float(x)
    except:
        try:
            return x.item()
        except:
            return str(x)

for gen, entry in enumerate(res.history):

    pop = entry.pop
    rank = pop.get("rank")
    crowd = pop.get("crowding")

    G = pop.get("G")
    H = pop.get("H")

    for i, ind in enumerate(entry.pop):

        row = {
            "generation": gen,
            "individual": i,
            "is_pareto": int(rank[i] == 0) if rank is not None else None,
            "crowding": (
                to_scalar(crowd[i])
                if (crowd is not None and crowd[i] is not None)
                else None
            )
        }

        # -------------------------
        # Decision variables
        # -------------------------
        if ind.X is not None:
            if isinstance(ind.X, dict):
                for k, v in ind.X.items():
                    row[f"X_{k}"] = to_scalar(v)
            else:
                for j, v in enumerate(ind.X):
                    row[f"X{j}"] = to_scalar(v)

        # -------------------------
        # Objectives
        # -------------------------
        if ind.F is not None:
            for j, val in enumerate(ind.F):
                row[f"F{j}"] = to_scalar(val)

        # -------------------------
        # Constraints G
        # -------------------------
        if G is not None:
            for j, val in enumerate(G[i]):
                row[f"G{j}"] = to_scalar(val)


        rows.append(row)

df = pl.DataFrame(rows)

df.write_csv("results.csv")


print("\nResults Saved")

# inspect execution time (optional)
hours = int(res.exec_time // 3600)
minutes = int((res.exec_time % 3600) // 60)
seconds = int((res.exec_time % 3600))

# 1 hour- 4 : 2by 2

print(f"\nExecution time: {hours}h {minutes}m {seconds}s")